In [1]:
import sys, time, json, csv, random, statistics, platform, datetime
from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort
import psutil
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

MODEL_NAME = "mobilenet_v2"          # later you can try "resnet18"
DATA = Path("../data/imagenette2-160")   # same data as Phases 3 and 4
OUT = Path("../models/phase6")
RESULTS = Path("../../results")
OUT.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

FP32_PATH = OUT / f"{MODEL_NAME}_fp32.onnx"
INT8_PATH = OUT / f"{MODEL_NAME}_int8.onnx"
print("Setup done.")

Setup done.


In [2]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_model(name):
    if name == "mobilenet_v2":
        m = models.mobilenet_v2(weights="DEFAULT")
    elif name == "resnet18":
        m = models.resnet18(weights="DEFAULT")
    else:
        raise ValueError(name)
    return m.eval()

In [4]:
TRIALS = 10          # how many repeated rounds
RUNS = 50            # timed runs inside each round
WARMUP = 10          # throwaway runs before timing
THREADS = [1, 4]
AFFINITY = "0-11"
TAG = "laptopA_legacy_fast"
USE_OPENVINO = False

def parse_cpus(s):
    cpus = []
    for part in s.split(","):
        if "-" in part:
            a, b = part.split("-")
            cpus += list(range(int(a), int(b) + 1))
        else:
            cpus.append(int(part))
    return cpus

if AFFINITY:
    cpus = parse_cpus(AFFINITY)
    psutil.Process().cpu_affinity(cpus)
    THREADS = [t for t in THREADS if t <= len(cpus)]
    print("Pinned to logical CPUs:", cpus)

try:
    import cpuinfo
    cpu_name = cpuinfo.get_cpu_info().get("brand_raw", platform.processor())
except Exception:
    cpu_name = platform.processor()

batt = psutil.sensors_battery()
info = {
    "tag": TAG, "model": MODEL_NAME, "cpu": cpu_name,
    "logical_cpus": psutil.cpu_count(logical=True),
    "os": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__, "onnxruntime": ort.__version__,
    "plugged_in": None if batt is None else batt.power_plugged,
    "affinity": AFFINITY, "trials": TRIALS, "runs_per_trial": RUNS,
}
print(json.dumps(info, indent=2))
if info["plugged_in"] is False:
    print("WARNING: on battery. Plug in and use best-performance power mode.")

x_np = np.random.default_rng(0).standard_normal((1, 3, 224, 224)).astype(np.float32)

Pinned to logical CPUs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
{
  "tag": "laptopA_legacy_fast",
  "model": "mobilenet_v2",
  "cpu": "13th Gen Intel(R) Core(TM) i7-13620H",
  "logical_cpus": 16,
  "os": "Windows-10-10.0.26200-SP0",
  "python": "3.11.15",
  "torch": "2.11.0+cu126",
  "onnxruntime": "1.30.0",
  "plugged_in": true,
  "affinity": "0-11",
  "trials": 10,
  "runs_per_trial": 50
}


In [5]:
import copy, warnings
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch_pruning as tp
warnings.filterwarnings("ignore")
assert MODEL_NAME == "mobilenet_v2"   # the repair below is specific to MobileNetV2

def masked_prune(m):                  # Phase 1 style: zeros weights, same size
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            prune.ln_structured(mod, name="weight", amount=0.2, n=2, dim=0)
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            prune.remove(mod, "weight")
    return m

def dyn_quant(m):                     # old-style dynamic quantization (classifier only)
    return torch.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)

def structural_prune(m):              # Phase 2 style: really removes channels
    tp.pruner.MagnitudePruner(
        m, torch.randn(1, 3, 224, 224),
        importance=tp.importance.MagnitudeImportance(p=2),
        pruning_ratio=0.2, ignored_layers=[m.classifier]).step()
    return m

def structural_q_then_p_repaired():   # same repair as your Phase 2 notebook
    pristine = load_model(MODEL_NAME)
    w = pristine.classifier[1].weight.detach().clone()
    b = pristine.classifier[1].bias.detach().clone()
    qm = dyn_quant(load_model(MODEL_NAME))
    pruner = tp.pruner.MagnitudePruner(
        qm, torch.randn(1, 3, 224, 224),
        importance=tp.importance.MagnitudeImportance(p=2),
        pruning_ratio=0.2, ignored_layers=[qm.classifier])
    pruner.step()
    removed = None
    for layer_name, is_out, idxs in pruner.pruning_history():
        if layer_name == "features.18.0":
            removed = idxs
            break
    kept = sorted(set(range(1280)) - set(removed))
    lin = nn.Linear(len(kept), 1000)
    with torch.no_grad():
        lin.weight.copy_(w[:, torch.tensor(kept)])
        lin.bias.copy_(b)
    lin.qconfig = torch.quantization.default_dynamic_qconfig
    qm.classifier = nn.Sequential(
        nn.Dropout(0.2), torch.ao.nn.quantized.dynamic.Linear.from_float(lin))
    return qm

N = MODEL_NAME
legacy = {
    "1_baseline_fp32":                 load_model(N),
    "2_masked_P":                      masked_prune(load_model(N)),
    "3_Q_dynamic":                     dyn_quant(load_model(N)),
    "4_masked_P_then_Q":               dyn_quant(masked_prune(load_model(N))),
    "5_masked_Q_then_P":               masked_prune(dyn_quant(load_model(N))),
    "6_structural_P":                  structural_prune(load_model(N)),
    "7_structural_P_then_Q":           dyn_quant(structural_prune(load_model(N))),
    "8_structural_Q_then_P_repaired":  structural_q_then_p_repaired(),
}
x_t = torch.from_numpy(x_np)
for name, m in legacy.items():
    with torch.inference_mode():
        assert m(x_t).shape == (1, 1000), name
    print("OK", name)

OK 1_baseline_fp32
OK 2_masked_P
OK 3_Q_dynamic
OK 4_masked_P_then_Q
OK 5_masked_Q_then_P
OK 6_structural_P
OK 7_structural_P_then_Q
OK 8_structural_Q_then_P_repaired


In [6]:
class Config:
    def __init__(self, runtime, precision, threads, prepare, run):
        self.runtime, self.precision, self.threads = runtime, precision, threads
        self.prepare, self.run = prepare, run
        self.name = f"{runtime}_{precision}_t{threads}"

configs = []
for name, m in legacy.items():
    for t in THREADS:
        def prepare(t=t):
            torch.set_num_threads(t)
        def run(m=m):
            with torch.inference_mode():
                m(x_t)
        configs.append(Config(name, "eager", t, prepare, run))
print(len(configs), "configs ready")

16 configs ready


In [8]:
def time_block(fn, runs):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000.0)   # milliseconds
    return statistics.median(times)

def bootstrap_ci(vals, n=2000, seed=0):
    r = random.Random(seed)
    meds = sorted(statistics.median(r.choices(vals, k=len(vals))) for _ in range(n))
    return meds[int(0.025 * n)], meds[int(0.975 * n) - 1]

In [10]:
print("Initial warm-up ...")
for c in configs:
    c.prepare()
    for _ in range(30):
        c.run()

trial_meds = {c.name: [] for c in configs}
rng = random.Random(0)
for trial in range(TRIALS):
    order = configs[:]
    rng.shuffle(order)            # mix the order so heat affects everyone equally
    for c in order:
        c.prepare()
        for _ in range(WARMUP):
            c.run()
        trial_meds[c.name].append(time_block(c.run, RUNS))
    print(f"trial {trial + 1}/{TRIALS} done")

Initial warm-up ...
trial 1/10 done
trial 2/10 done
trial 3/10 done
trial 4/10 done
trial 5/10 done
trial 6/10 done
trial 7/10 done
trial 8/10 done
trial 9/10 done
trial 10/10 done


In [11]:
rows = []
for c in configs:
    v = trial_meds[c.name]
    lo, hi = bootstrap_ci(v)
    rows.append({"model": c.runtime, "threads": c.threads,
                 "median_ms": statistics.median(v), "ci_low_ms": lo, "ci_high_ms": hi,
                 "cv_percent": 100 * statistics.pstdev(v) / statistics.mean(v)})

def overlaps(a, b):
    return not (a["ci_high_ms"] < b["ci_low_ms"] or a["ci_low_ms"] > b["ci_high_ms"])

base = {r["threads"]: r for r in rows if r["model"] == "1_baseline_fp32"}
for r in rows:
    b = base[r["threads"]]
    r["x_vs_baseline"] = b["median_ms"] / r["median_ms"]
    r["verdict"] = ("reference" if r is b else
                    "within noise" if overlaps(r, b) else
                    "faster" if r["median_ms"] < b["median_ms"] else "slower")

print(f"{'model':34s} {'thr':>3s} {'median ms':>10s} {'95% CI':>18s} {'CV%':>6s} {'x base':>7s}  verdict")
for r in sorted(rows, key=lambda r: (r["threads"], r["model"])):
    ci = f"[{r['ci_low_ms']:.2f}, {r['ci_high_ms']:.2f}]"
    print(f"{r['model']:34s} {r['threads']:3d} {r['median_ms']:10.2f} {ci:>18s} "
          f"{r['cv_percent']:6.1f} {r['x_vs_baseline']:7.2f}  {r['verdict']}")

def compare(a, b, t):
    ra = next(r for r in rows if r["model"] == a and r["threads"] == t)
    rb = next(r for r in rows if r["model"] == b and r["threads"] == t)
    print(f"[{t} thr] {a} {ra['median_ms']:.2f} ms vs {b} {rb['median_ms']:.2f} ms -> "
          f"{'within noise' if overlaps(ra, rb) else 'DIFFERENT'}")

print("\nThe sequencing question:")
for t in THREADS:
    compare("4_masked_P_then_Q", "5_masked_Q_then_P", t)
    compare("7_structural_P_then_Q", "8_structural_Q_then_P_repaired", t)

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
stem = RESULTS / f"phase6b_legacy_{TAG}_{stamp}"
with open(f"{stem}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
with open(f"{stem}.json", "w") as f:
    json.dump(info, f, indent=2)
print("Saved", stem)

model                              thr  median ms             95% CI    CV%  x base  verdict
1_baseline_fp32                      1      23.06     [21.84, 23.60]   12.1    1.00  reference
2_masked_P                           1      23.29     [22.48, 23.50]    3.8    0.99  within noise
3_Q_dynamic                          1      23.21     [22.75, 23.95]    3.9    0.99  within noise
4_masked_P_then_Q                    1      23.08     [21.08, 23.58]    6.5    1.00  within noise
5_masked_Q_then_P                    1      23.21     [22.90, 24.11]    2.7    0.99  within noise
6_structural_P                       1      18.28     [17.58, 18.73]   10.2    1.26  faster
7_structural_P_then_Q                1      18.70     [17.94, 18.88]    4.8    1.23  faster
8_structural_Q_then_P_repaired       1      18.78     [18.14, 19.33]    3.4    1.23  faster
1_baseline_fp32                      4      17.78     [15.04, 18.41]   16.7    1.00  reference
2_masked_P                           4      17.57